In [1]:
import sys
import os
import mysql.connector
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Ambil konfigurasi database dari folder utama project kalian
sys.path.append(os.path.abspath('..'))
from config import get_db_config

config = get_db_config()

# 1. Koneksi ke Database Baru (Fase Migrasi Sekarang)
db_new = mysql.connector.connect(**config['db_new'])
cursor_new = db_new.cursor(dictionary=True)

# 2. Koneksi ke Database Masa Depan (DB_FUTURE)
# Catatan: Pastikan di file config.py kalian sudah ada key 'db_future' ya!
db_future = mysql.connector.connect(**config['db_future'])
cursor_future = db_future.cursor(dictionary=True)

print(f"✅ Sukses Terhubung ke Database Baru : {config['db_new']['database']}")
print(f"🚀 Sukses Terhubung ke DB_FUTURE     : {config['db_future']['database']}")

✅ Sukses Terhubung ke Database Baru : 3
🚀 Sukses Terhubung ke DB_FUTURE     : 3


In [2]:
tables_to_check = [
    # --- Bagian Cimut (Fokus Utama) ---
    "kontak_prospek", 
    "calon_siswa", 
    "calon_siswa_akademik", 
    "calon_siswa_ortu", 
    "calon_siswa_bayar", 
    "calon_siswa_jadwal", 
    "calon_siswa_kursus", 
    "calon_siswa_proses", 
    "calon_siswa_status_logs", 
    "peminjaman", 
    "pengadaan", 
    "problem",
    
    # --- Bagian Afrida ---
    "sop", 
    "surat_keluar", 
    "verifikasi_surat_keluar", 
    "surat_tugas", 
    "surat_tugas_anggota",
    
    # --- Bagian Hanif ---
    "pengajuan_karyawan", 
    "histori_pengajuan", 
    "pelamar", 
    "pelamar_kerja", 
    "pelamar_sekolah", 
    "pelamar_kursus", 
    "progres_pelamar", 
    "rekrutmen_pelamar"
]

In [3]:
# === Cell 2: Inspeksi Detektor Pintar dengan Prioritas Target Revisi di Atas ===
import pandas as pd
import numpy as np

print("================================================================================")
print(" 🕵️‍♂️ FORENSIK ARSITEKTUR KONTROL: DETEKTOR AUTO-SMART COMPILER (REVISI PRIORITY) 🕵️‍♂️ ")
print("================================================================================")

# List penampung data di memori untuk keperluan sorting visualisasi
revisi_tables_queue = []
identical_tables_queue = []

# --- TAHAP A: PROSES PEN ARIKAN DATA & EVALUASI STRUKTUR DI BELAKANG LAYAR ---
for table in tables_to_check:
    try:
        # 1. Ambil data asli dari DB_NEW untuk kebutuhan .info() dan sampel isi data
        query = f"SELECT * FROM `{table}`"
        df_real_data = pd.read_sql(query, db_new)
        
        # 2. Tarik Struktur Fisik Kolom dari DB_NEW
        query_new_struct = f"""
        SELECT 
            c.COLUMN_NAME AS 'Nama Kolom',
            c.COLUMN_TYPE AS 'Tipe Data MySQL',
            CONCAT(
                CASE WHEN c.IS_NULLABLE = 'YES' THEN '✅ NULL (Boleh Kosong)' ELSE '🛑 NOT NULL (Wajib Isi)' END,
                CASE WHEN c.EXTRA = 'auto_increment' THEN '\\n🚀 AUTO_INCREMENT' ELSE '' END
            ) AS 'Aturan Nullability & Increment',
            CASE 
                WHEN c.COLUMN_KEY = 'PRI' THEN '🔑 PRIMARY KEY (PK)'
                WHEN c.COLUMN_KEY = 'MUL' AND k.REFERENCED_TABLE_NAME IS NOT NULL THEN '🔗 FOREIGN KEY (FK)'
                WHEN c.COLUMN_KEY = 'MUL' THEN 'INDEX'
                ELSE '-'
            END AS 'Status Kunci',
            CASE 
                WHEN k.REFERENCED_TABLE_NAME IS NOT NULL THEN CONCAT(k.REFERENCED_TABLE_NAME, ' (', k.REFERENCED_COLUMN_NAME, ')')
                ELSE '-'
            END AS 'Rujukan Induk (FK Origin)',
            CASE WHEN c.DATA_TYPE = 'enum' THEN REPLACE(REPLACE(REPLACE(c.COLUMN_TYPE, 'enum(', ''), ')', ''), "'", "") ELSE '-' END AS 'Daftar Pilihan ENUM'
        FROM INFORMATION_SCHEMA.COLUMNS c
        LEFT JOIN INFORMATION_SCHEMA.KEY_COLUMN_USAGE k 
            ON c.TABLE_SCHEMA = k.TABLE_SCHEMA AND c.TABLE_NAME = k.TABLE_NAME AND c.COLUMN_NAME = k.COLUMN_NAME
        WHERE c.TABLE_SCHEMA = '{config['db_new']['database']}' AND c.TABLE_NAME = '{table}'
        ORDER BY c.ORDINAL_POSITION;
        """
        df_struct_new = pd.read_sql(query_new_struct, db_new).drop_duplicates(subset=['Nama Kolom'], keep='first').reset_index(drop=True)
        
        # Lacak tabel anak yang nge-FK ke PK tabel ini di DB_NEW
        pk_referenced_list = []
        for idx, row_skri in df_struct_new.iterrows():
            if row_skri['Status Kunci'] == '🔑 PRIMARY KEY (PK)':
                lookup_fk_query = f"""
                SELECT CONCAT(TABLE_NAME, ' (', COLUMN_NAME, ')') as relasi
                FROM INFORMATION_SCHEMA.KEY_COLUMN_USAGE
                WHERE REFERENCED_TABLE_SCHEMA = '{config['db_new']['database']}'
                  AND REFERENCED_TABLE_NAME = '{table}'
                  AND REFERENCED_COLUMN_NAME = '{row_skri['Nama Kolom']}';
                """
                df_relasi_luar = pd.read_sql(lookup_fk_query, db_new)
                if not df_relasi_luar.empty:
                    pk_referenced_list.append("\n".join(df_relasi_luar['relasi'].tolist()))
                else:
                    pk_referenced_list.append("-")
            else:
                pk_referenced_list.append("-")
        df_struct_new['Tabel Yang nge-FK (DB_NEW)'] = pk_referenced_list

        # 3. Tarik Struktur Fisik Kolom dari DB_FUTURE
        query_future_struct = f"""
        SELECT 
            c.COLUMN_NAME AS 'Nama Kolom',
            c.COLUMN_TYPE AS 'Tipe Data MySQL',
            CONCAT(
                CASE WHEN c.IS_NULLABLE = 'YES' THEN '✅ NULL (Boleh Kosong)' ELSE '🛑 NOT NULL (Wajib Isi)' END,
                CASE WHEN c.EXTRA = 'auto_increment' THEN '\\n🚀 AUTO_INCREMENT' ELSE '' END
            ) AS 'Aturan Nullability & Increment',
            CASE 
                WHEN c.COLUMN_KEY = 'PRI' THEN '🔑 PRIMARY KEY (PK)'
                WHEN c.COLUMN_KEY = 'MUL' AND k.REFERENCED_TABLE_NAME IS NOT NULL THEN '🔗 FOREIGN KEY (FK)'
                WHEN c.COLUMN_KEY = 'MUL' THEN 'INDEX'
                ELSE '-'
            END AS 'Status Kunci',
            CASE 
                WHEN k.REFERENCED_TABLE_NAME IS NOT NULL THEN CONCAT(k.REFERENCED_TABLE_NAME, ' (', k.REFERENCED_COLUMN_NAME, ')')
                ELSE '-'
            END AS 'Rujukan Induk (FK Origin)',
            CASE WHEN c.DATA_TYPE = 'enum' THEN REPLACE(REPLACE(REPLACE(c.COLUMN_TYPE, 'enum(', ''), ')', ''), "'", "") ELSE '-' END AS 'Daftar Pilihan ENUM'
        FROM INFORMATION_SCHEMA.COLUMNS c
        LEFT JOIN INFORMATION_SCHEMA.KEY_COLUMN_USAGE k 
            ON c.TABLE_SCHEMA = k.TABLE_SCHEMA AND c.TABLE_NAME = k.TABLE_NAME AND c.COLUMN_NAME = k.COLUMN_NAME
        WHERE c.TABLE_SCHEMA = '{config['db_future']['database']}' AND c.TABLE_NAME = '{table}'
        ORDER BY c.ORDINAL_POSITION;
        """
        df_struct_future = pd.read_sql(query_future_struct, db_future).drop_duplicates(subset=['Nama Kolom'], keep='first').reset_index(drop=True)
        
        # Lacak tabel anak yang nge-FK ke PK tabel ini di DB_FUTURE
        pk_referenced_list_future = []
        for idx, row_skri in df_struct_future.iterrows():
            if row_skri['Status Kunci'] == '🔑 PRIMARY KEY (PK)':
                lookup_fk_query_future = f"""
                SELECT CONCAT(TABLE_NAME, ' (', COLUMN_NAME, ')') as relasi
                FROM INFORMATION_SCHEMA.KEY_COLUMN_USAGE
                WHERE REFERENCED_TABLE_SCHEMA = '{config['db_future']['database']}'
                  AND REFERENCED_TABLE_NAME = '{table}'
                  AND REFERENCED_COLUMN_NAME = '{row_skri['Nama Kolom']}';
                """
                df_relasi_luar_future = pd.read_sql(lookup_fk_query_future, db_future)
                if not df_relasi_luar_future.empty:
                    pk_referenced_list_future.append("\n".join(df_relasi_luar_future['relasi'].tolist()))
                else:
                    pk_referenced_list_future.append("-")
            else:
                pk_referenced_list_future.append("-")
        df_struct_future['Tabel Yang nge-FK (DB_FUTURE)'] = pk_referenced_list_future

        # 4. Deep Comparison Kesamaan Jeroan Kolom dasar
        cols_to_compare = ['Nama Kolom', 'Tipe Data MySQL', 'Aturan Nullability & Increment', 'Status Kunci', 'Rujukan Induk (FK Origin)', 'Daftar Pilihan ENUM']
        
        is_structure_identical = False
        if not df_struct_new.empty and not df_struct_future.empty:
            is_structure_identical = df_struct_new[cols_to_compare].equals(df_struct_future[cols_to_compare])

        # Wadah paket data tabel untuk di-render nanti
        table_package = {
            'name': table,
            'df_real_data': df_real_data,
            'df_struct_new': df_struct_new,
            'df_struct_future': df_struct_future,
            'is_identical': is_structure_identical
        }

        # 🔥 FILTER SAKTI CIMUT: Pisahkan antrean, utamakan yang bermasalah (revisi) ke atas!
        if is_structure_identical:
            identical_tables_queue.append(table_package)
        else:
            revisi_tables_queue.append(table_package)
            
    except Exception as e:
        print(f"❌ Gagal menganalisis awal tabel `{table}`: {e}")

# --- TAHAP B: MULAI PEN TAMPILAN VISUALISASI BERDASARKAN ANT REAN PRIORITAS ---

# 🚨 1. KELOMPOK UTAMA (PALING ATAS): DAFTAR TABEL YANG WAJIB DIREVISI 🚨
if revisi_tables_queue:
    print("\n" + "!"*80)
    print(f"🚨 [🔥 REVISI PRIORITY BOARD] TERDETEKSI {len(revisi_tables_queue)} TABEL BERBEDA - HARUS SEGERA DIPERBAIKI!")
    print("!"*80)
    
    for pkg in revisi_tables_queue:
        print(f"\n================================================================================")
        print(f"⚠️  [STATUS: TARGET REVISI] TABEL: {pkg['name'].upper()}")
        print(f"================================================================================")
        
        print(f"📋 1. Karakteristik Wadah Pandas Dataframe (.info()):")
        pkg['df_real_data'].info()
        print("\n" + "-"*60)
        
        print(f"🔍 2. [TABEL REAL SAAT INI] Arsitektur Fisik Kolom pada DB_NEW:")
        display(pkg['df_struct_new'].style.set_properties(**{'white-space': 'pre-wrap', 'text-align': 'left'}))
        print("\n" + "-"*60)
        
        print(f"🔮 3. [⚠️ TARGET BLUEPRINT RUJUKAN] Tinjau Struktur yang Benar pada DB_FUTURE di bawah:")
        if pkg['df_struct_future'].empty:
            print("❌ ERROR: Tabel ini tidak ditemukan / belum dibuat sama sekali di DB_FUTURE!")
        else:
            display(pkg['df_struct_future'].style.set_properties(**{'white-space': 'pre-wrap', 'text-align': 'left'}))
        print("\n" + "-"*60)
        
        print(f"📸 4. Sampel Isi Data Real di DB_NEW:")
        display(pkg['df_real_data'] if pkg['df_real_data'].empty else pkg['df_real_data'])
        print("\n" + "="*80)

# ✨ 2. KELOMPOK KEDUA (BAW AH): DAFTAR TABEL YANG SUDAH AMAN IDENTIK ✨
if identical_tables_queue:
    print("\n" + "="*80)
    print(f"✨ [🟢 PERFECTLY MATCHED BOARD] SEBANYAK {len(identical_tables_queue)} TABEL STRUKTURNYA SUDAH IDENTIK SAMA PERSIS!")
    print("="*80)
    
    for pkg in identical_tables_queue:
        print(f"\n================================================================================")
        print(f"✅ [STATUS: AMAN IDENTIK] TABEL: {pkg['name'].upper()}")
        print(f"================================================================================")
        
        print(f"📋 1. Karakteristik Wadah Pandas Dataframe (.info()):")
        pkg['df_real_data'].info()
        print("\n" + "-"*60)
        
        print(f"🔍 2. [TABEL REAL & FINAL] Arsitektur Fisik Kolom pada DB_NEW:")
        display(pkg['df_struct_new'].style.set_properties(**{'white-space': 'pre-wrap', 'text-align': 'left'}))
        print("\n" + "-"*60)
        
        print("✨ INDICATOR: [ DB_NEW ] & [ DB_FUTURE ] STRUKTUR SUDAH IDENTIK SAMA PERSIS 100% ✨")
        print("ℹ️  Tabel DB_FUTURE otomatis disembunyikan untuk kenyamanan pemandangan layar monitor.")
        print("\n" + "-"*60)
        
        print(f"📸 3. Sampel Isi Data Real di DB_NEW:")
        display(pkg['df_real_data'] if pkg['df_real_data'].empty else pkg['df_real_data'])
        print("\n" + "="*80)

 🕵️‍♂️ FORENSIK ARSITEKTUR KONTROL: DETEKTOR AUTO-SMART COMPILER (REVISI PRIORITY) 🕵️‍♂️ 



✨ [🟢 PERFECTLY MATCHED BOARD] SEBANYAK 25 TABEL STRUKTURNYA SUDAH IDENTIK SAMA PERSIS!

✅ [STATUS: AMAN IDENTIK] TABEL: KONTAK_PROSPEK
📋 1. Karakteristik Wadah Pandas Dataframe (.info()):
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 194 entries, 0 to 193
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   id_kontak_prospek        194 non-null    int64         
 1   kode_kontak              194 non-null    object        
 2   nama_penanya             194 non-null    object        
 3   nomor_telepon            194 non-null    object        
 4   email                    163 non-null    object        
 5   sumber_informasi         165 non-null    object        
 6   catatan_awal_fo          0 non-null      object        
 7   id_admin_fo              0 non-null      object        
 8   status_kontak            194 non-null    object        
 9   tanggal_kontak_pertama   0 non

,Nama Kolom,Tipe Data MySQL,Aturan Nullability & Increment,Status Kunci,Rujukan Induk (FK Origin),Daftar Pilihan ENUM,Tabel Yang nge-FK (DB_NEW)
0,id_kontak_prospek,bigint(20) unsigned,🛑 NOT NULL (Wajib Isi) 🚀 AUTO_INCREMENT,🔑 PRIMARY KEY (PK),-,-,calon_siswa (id_kontak_prospek)
1,kode_kontak,varchar(50),🛑 NOT NULL (Wajib Isi),-,-,-,-
2,nama_penanya,varchar(150),🛑 NOT NULL (Wajib Isi),-,-,-,-
3,nomor_telepon,varchar(20),🛑 NOT NULL (Wajib Isi),-,-,-,-
4,email,varchar(150),✅ NULL (Boleh Kosong),-,-,-,-
5,sumber_informasi,varchar(100),✅ NULL (Boleh Kosong),-,-,-,-
6,catatan_awal_fo,text,✅ NULL (Boleh Kosong),-,-,-,-
7,id_admin_fo,varchar(15),✅ NULL (Boleh Kosong),🔗 FOREIGN KEY (FK),users (id_user),-,-
8,status_kontak,varchar(50),🛑 NOT NULL (Wajib Isi),-,-,-,-
9,tanggal_kontak_pertama,timestamp,✅ NULL (Boleh Kosong),-,-,-,-



------------------------------------------------------------
✨ INDICATOR: [ DB_NEW ] & [ DB_FUTURE ] STRUKTUR SUDAH IDENTIK SAMA PERSIS 100% ✨
ℹ️  Tabel DB_FUTURE otomatis disembunyikan untuk kenyamanan pemandangan layar monitor.

------------------------------------------------------------
📸 3. Sampel Isi Data Real di DB_NEW:


,id_kontak_prospek,kode_kontak,nama_penanya,nomor_telepon,email,sumber_informasi,catatan_awal_fo,id_admin_fo,status_kontak,tanggal_kontak_pertama,tanggal_kontak_terakhir
0,1,PQ8WD1T7,Daria Azmiya Jasmine,082231346758,puterihapsari.f@gmail.com,Teman/kerabat/saudara,None,None,1,None,2025-09-18 17:39:40
1,2,FRX3MYBA,Annisa Zahro Ramadhania,081336647476,dwirohm4@gmail.com,Instagram,None,None,1,None,2025-09-23 16:25:14
2,3,8SZ2FCXU,Zulfa Bariatur Rahma,085707179656,chyzryth@gmail.com,Lainnya,None,None,1,None,2025-10-03 09:48:34
3,4,V4LATPWN,Achmad Naufal Albiruni,087765283592,melisnifuku@gmail.com,Instagram,None,None,1,None,2025-10-03 09:49:15
4,5,D81NEPDX,Khansa Amalia Putri Aji,081554932188,afadhilpa@gmail.com,Teman/kerabat/saudara,None,None,0,None,2025-10-03 09:48:50
...,...,...,...,...,...,...,...,...,...,...,...
189,190,ROLGGUVW,Atiya Sabita,,None,None,None,None,waiting for confirmation,None,2025-10-17 15:41:09
190,191,KIUULWZE,Dwi,,None,None,None,None,follow up another time,None,2025-10-17 15:41:26
191,192,5PBZSF6H,Aca,,None,None,None,None,follow up another time,None,2025-10-24 15:27:54
192,193,OHGUROZF,Bu Hartik,,None,None,None,None,waiting for confirmation,None,2025-10-24 15:28:21


...

,id_progres_pelamar,id_pelamar,id_user,status_progres_pelamar,catatan,tautan_file,pertanyaan,created_at
0,14,178,U00001,Tahap Test,<p>interview</p>,https://drive.google.com/drive/folders/1WYB9iR...,,2023-05-29 17:45:09
1,15,178,U00001,Interview,,,,2023-05-30 06:21:30
2,18,178,U00011,Interview,<p>haha</p>,,,2023-06-12 14:09:10
3,19,2,U00018,Baru,<p>Pelamar bersedia melaksanakan tugas dan jad...,,<p>Pelamar bersedia melaksanakan tugas dan jad...,2023-07-02 19:45:07
4,25,2,U00018,Interview,<p>jadi jawabnya seperti ini</p>,,<p><strong>Informasi dasar yang perlu digali (...,2023-07-02 20:46:56
...,...,...,...,...,...,...,...,...
283,456,174,U00018,Tahap Test,,https://drive.google.com/drive/folders/1e7QK2s...,,2026-04-08 08:22:57
284,457,175,U00018,Tahap Test,,https://drive.google.com/drive/folders/1Ofqg2q...,,2026-04-08 08:24:43
285,458,176,U00018,Tahap Test,,https://drive.google.com/drive/folders/1fYg26w...,,2026-04-13 09:24:34
286,459,176,U00018,Ditolak,,,,2026-04-17 10:44:29




✅ [STATUS: AMAN IDENTIK] TABEL: REKRUTMEN_PELAMAR
📋 1. Karakteristik Wadah Pandas Dataframe (.info()):
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 205 entries, 0 to 204
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id_rekrutmen  205 non-null    int64  
 1   id_pelamar    193 non-null    float64
 2   id_user       205 non-null    object 
dtypes: float64(1), int64(1), object(1)
memory usage: 4.9+ KB

------------------------------------------------------------
🔍 2. [TABEL REAL & FINAL] Arsitektur Fisik Kolom pada DB_NEW:


,Nama Kolom,Tipe Data MySQL,Aturan Nullability & Increment,Status Kunci,Rujukan Induk (FK Origin),Daftar Pilihan ENUM,Tabel Yang nge-FK (DB_NEW)
0,id_rekrutmen,bigint(20) unsigned,🛑 NOT NULL (Wajib Isi) 🚀 AUTO_INCREMENT,🔑 PRIMARY KEY (PK),-,-,-
1,id_pelamar,bigint(20) unsigned,✅ NULL (Boleh Kosong),🔗 FOREIGN KEY (FK),pelamar (id_pelamar),-,-
2,id_user,varchar(15),✅ NULL (Boleh Kosong),🔗 FOREIGN KEY (FK),users (id_user),-,-



------------------------------------------------------------
✨ INDICATOR: [ DB_NEW ] & [ DB_FUTURE ] STRUKTUR SUDAH IDENTIK SAMA PERSIS 100% ✨
ℹ️  Tabel DB_FUTURE otomatis disembunyikan untuk kenyamanan pemandangan layar monitor.

------------------------------------------------------------
📸 3. Sampel Isi Data Real di DB_NEW:


,id_rekrutmen,id_pelamar,id_user
0,12,2.0,U00014
1,13,2.0,U00023
2,20,12.0,U00014
3,21,12.0,U00016
4,24,NaN,U00020
...,...,...,...
200,370,173.0,U00015
201,371,173.0,U00018
202,372,176.0,U00014
203,373,176.0,U00015
